Getting Started with Machine Learning: Classification

# What is Machine Learning
  ML is a subset of Artificial Intelligence focused on algorithms that can understand and analyze patterns off training data in order to make accurate inferences about new data. This enables machine learning models to make decisions or predictions without explicit instructions.

# Supervised vs Unsupervised Learning

=========================================

# Supervised Learning
An approach where the model is trained to predict the "correct" putput for a given input. This technique uses labeled dataset to train the model to identify underlying patterns and relationships.

# Unsupervised Learning
Uses ML algorithms to analyze and cluster unlabeled datasets. These algorithms discover hidden patterns and data groupings without the need for human intervention.

For this project, I will be focusing on Classificaiton algorithms to answer the question **"What evacuation-relevant variables consistently matter across multiple disasters?"**, with the help of Titanic Survival dataset and other similar datasets to be used for comparison.

# Step 1. Load Raw dataset
In order to visualize the contents of the dataset you'll be working on, this will also serve as a comparison point once the dataset has been cleaned

In [7]:

import pandas as pd

df= pd.read_csv('Titanic-Dataset.csv')
df.head(20)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [8]:
df2= pd.read_csv('LusitaniaManifest.csv')
df2.head(20)

,Unnamed: 0,Family name,Title,Personal name,Fate,Age,Department/Class,Passenger/Crew,Citizenship,Position,Status,City,Lifeboat,Rescue Vessel,Adult/Minor,Sex
0,0,CAMERON,Mr.,Charles W.,Lost,38.0,Band,Crew,British,NaN,NaN,NaN,NaN,NaN,Adult,Male
1,1,CARR-JONES,Mr.,E.,Lost,37.0,Band,Crew,British,NaN,NaN,NaN,NaN,NaN,Adult,Male
2,2,DRAKEFORD,Mr.,Edward,Saved,30.0,Band,Crew,British,Violin,NaN,NaN,NaN,NaN,Adult,Male
3,3,HAWKINS,Mr.,Handel,Saved,25.0,Band,Crew,British,Cello,NaN,NaN,NaN,NaN,Adult,Male
4,4,HEMINGWAY,Mr.,John William,Saved,27.0,Band,Crew,British,Double Bass,NaN,NaN,NaN,NaN,Adult,Male
5,5,ANDERSON,Mr.,James Clarke,Lost,48.0,Deck,Crew,British,Staff Captain,NaN,Liverpool,NaN,NaN,Adult,Male
6,6,ANDERSON,Mr.,John,Lost,NaN,Deck,Crew,Norwegian,Able-Bodied Seaman,NaN,NaN,NaN,NaN,Adult,Male
7,7,BATTLE,Mr.,James,Saved,NaN,Deck,Crew,British,Able-Bodied Seaman,NaN,Sligo,Found Lifeboat,Found by Rescue Vessel,Adult,Male
8,8,BESTIC,Mr.,Albert Arthur,Saved,24.0,Deck,Crew,British,Junior Third Officer,Single,Dublin,NaN,Found by Rescue Vessel,Adult,Male
9,9,BOWDEN,Mr.,Joseph,Saved,19.0,Deck,Crew,British,Able-Bodied Seaman,NaN,NaN,NaN,NaN,Adult,Male


# Step 2. Filtering
Keep only relevant columns, any columns that are duplicated or can be derived from other columns may be dropped

# Justifications for filtering
Columns Kept:
   - survived: target
   - pclass, fare: socio-economic proxies
  - sex, age: social-norm / "women and children first" proxies
   - sibsp, parch: family-group proxies (does traveling with family change behavior?)
   - embarked/embark_town: kept for now, low priority hypothesis

 Dropped: 'deck' (77% missing, low reliability), 'class'/'who'/'alive'/'adult_male'
 (redundant duplicates of pclass/sex/survived already in the seaborn version),
 'alone' (derivable from sibsp+parch, kept implicit rather than duplicated).


In [9]:
keep_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
filtered = df[keep_cols].copy()
# Drop rows with no target label (shouldn't exist here, but always check)
filtered = filtered.dropna(subset=["Survived"])

# Age has ~20% missing -- for this exploratory stage we keep NaNs as-is
# rather than imputing, so grouping/aggregation functions can decide
# how to handle them explicitly (imputation belongs in a later modeling step,
# not the exploration step, so we don't quietly bias the groupings).

# Drop the 2 rows missing 'embarked' (negligible loss, not worth imputing)
filtered = filtered.dropna(subset=["Embarked"])

print()
print("=" * 70)
print("FILTERED SHAPE:", filtered.shape)
print("=" * 70)
print(filtered.head(10))

# Engineer one derived variable now, since it maps directly to a hypothesis:
# family_size = traveling with others vs. alone
filtered["family_size"] = filtered["SibSp"] + filtered["Parch"] + 1
filtered["traveling_alone"] = filtered["family_size"] == 1



FILTERED SHAPE: (889, 8)
   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0         0       3    male  22.0      1      0   7.2500        S
1         1       1  female  38.0      1      0  71.2833        C
2         1       3  female  26.0      0      0   7.9250        S
3         1       1  female  35.0      1      0  53.1000        S
4         0       3    male  35.0      0      0   8.0500        S
5         0       3    male   NaN      0      0   8.4583        Q
6         0       1    male  54.0      0      0  51.8625        S
7         0       3    male   2.0      3      1  21.0750        S
8         1       3  female  27.0      0      2  11.1333        S
9         1       2  female  14.0      1      0  30.0708        C


# Justifications for filtering
Columns Kept:
   - Fate: Equivalent to Survived col in titanic
   - sex: Core hypothesis variable, matches sex col in titanic
  -  age: Core hypothesis variable
   - adult/minor: backup for age, no missing values means it can fill the gap where age is missing
   -  department/class: sioci-economic proxy
   - passenger/crew: Critical — Titanic's dataset is passengers only, so you must filter or flag crew separately for a fair comparison

 Dropped:
 * Citizenship
 * Family name, Title, Personal name
 * Position
 * Status
 * City
 * Lifeboat
 * Rescue Vessel
    


In [10]:
keep_cols2 = ["Fate", "Sex", "Age", "Adult/Minor", "Department/Class", "Passenger/Crew"]
filtered2 = df2[keep_cols2].copy()
# Drop rows with no target label (Fate) if any exist
filtered2 = filtered2.dropna(subset=["Fate"])

# Removed lines for 'Embarked', 'SibSp', 'Parch' as they don't exist in Lusitania dataset

print()
print("=" * 70)
print("FILTERED2 SHAPE:", filtered2.shape)
print("=" * 70)
print(filtered2.head(10))



FILTERED2 SHAPE: (1961, 6)
    Fate   Sex   Age Adult/Minor Department/Class Passenger/Crew
0   Lost  Male  38.0       Adult             Band           Crew
1   Lost  Male  37.0       Adult             Band           Crew
2  Saved  Male  30.0       Adult             Band           Crew
3  Saved  Male  25.0       Adult             Band           Crew
4  Saved  Male  27.0       Adult             Band           Crew
5   Lost  Male  48.0       Adult             Deck           Crew
6   Lost  Male   NaN       Adult             Deck           Crew
7  Saved  Male   NaN       Adult             Deck           Crew
8  Saved  Male  24.0       Adult             Deck           Crew
9  Saved  Male  19.0       Adult             Deck           Crew


Note: This step only filtered out unused/irrelevant columns

# Partial step. Pooling
Pool the two filtered datasets into one unified DataFrame. Align common columns: survival status, Sex, Age
note: Pooling should be done once thedatasets are actually cleaned, this is only to show both datasets into one dataframe

In [14]:
# Pool the two filtered datasets into one unified DataFrame
# Align common columns: survival status, Sex, Age

# --- Prepare Titanic ---
titanic_pool = df[["Survived", "Sex", "Age", "Pclass", "Fare"]].copy()
titanic_pool["survived"] = titanic_pool["Survived"] == 1  # bool
titanic_pool["dataset"] = "Titanic"
titanic_pool["class"] = titanic_pool["Pclass"]
titanic_pool.drop(columns=["Survived", "Pclass"], inplace=True)

# --- Prepare Lusitania ---
lusitania_pool = filtered2[["Fate", "Sex", "Age", "Department/Class", "Adult/Minor"]].copy()
lusitania_pool["survived"] = lusitania_pool["Fate"] == "Saved"  # bool
lusitania_pool["dataset"] = "Lusitania"
lusitania_pool["class"] = lusitania_pool["Department/Class"]
lusitania_pool.drop(columns=["Fate", "Department/Class"], inplace=True)

# --- Combine ---
pooled = pd.concat([titanic_pool, lusitania_pool], ignore_index=True)

print()
print("=" * 70)
print("POOLED SHAPE:", pooled.shape)
print("=" * 70)
print(pooled.head(1000))
print()
print("Survival rate by dataset:")
print(pooled.groupby("dataset")["survived"].mean())


POOLED SHAPE: (2852, 7)
        Sex   Age     Fare  survived    dataset        class Adult/Minor
0      male  22.0   7.2500     False    Titanic            3         NaN
1    female  38.0  71.2833      True    Titanic            1         NaN
2    female  26.0   7.9250      True    Titanic            3         NaN
3    female  35.0  53.1000      True    Titanic            1         NaN
4      male  35.0   8.0500     False    Titanic            3         NaN
..      ...   ...      ...       ...        ...          ...         ...
995    Male   NaN      NaN      True  Lusitania  Engineering       Adult
996    Male  59.0      NaN     False  Lusitania  Engineering       Adult
997    Male   NaN      NaN      True  Lusitania  Engineering       Adult
998    Male   NaN      NaN     False  Lusitania  Engineering       Adult
999    Male  53.0      NaN     False  Lusitania  Engineering       Adult

[1000 rows x 7 columns]

Survival rate by dataset:
dataset
Lusitania    0.391637
Titanic      0.38

# Step 3. Data Cleaning
Process of preparing raw data by correcting errors so that it can be used effectively for analysis. Common data anomalies may include: Missing Values, Duplicaed records, Incorrect datatypes, Outliers, and Inconsistent formats

3.1 Cleaning Missing Values

In [19]:
# --- Clean Titanic dataset --- start with missing values ---
titanic_clean = filtered.copy()

print("Missing values before cleaning:")
print(titanic_clean.isnull().sum())
print()

# Age: impute with median grouped by Pclass + Sex
age_medians = titanic_clean.groupby(["Pclass", "Sex"])["Age"].median()
titanic_clean["Age"] = titanic_clean.apply(
    lambda row: age_medians.loc[(row["Pclass"], row["Sex"])]
    if pd.isna(row["Age"])
    else row["Age"],
    axis=1,
)

# Fare: 1 missing row, impute with median Pclass
fare_median = titanic_clean.groupby("Pclass")["Fare"].transform("median")
titanic_clean["Fare"] = titanic_clean["Fare"].fillna(fare_median)

# Embarked: 2 missing rows, drop since negligible
titanic_clean = titanic_clean.dropna(subset=["Embarked"])


print(titanic_clean.head(1000))

Missing values before cleaning:
Survived             0
Pclass               0
Sex                  0
Age                177
SibSp                0
Parch                0
Fare                 0
Embarked             0
family_size          0
traveling_alone      0
dtype: int64

     Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked  \
0           0       3    male  22.0      1      0   7.2500        S   
1           1       1  female  38.0      1      0  71.2833        C   
2           1       3  female  26.0      0      0   7.9250        S   
3           1       1  female  35.0      1      0  53.1000        S   
4           0       3    male  35.0      0      0   8.0500        S   
..        ...     ...     ...   ...    ...    ...      ...      ...   
886         0       2    male  27.0      0      0  13.0000        S   
887         1       1  female  19.0      0      0  30.0000        S   
888         0       3  female  21.5      1      2  23.4500        S   
889         1 

3.2 Cleaning Duplicated Values

In [20]:
# --- Investigate duplicates in Titanic dataset ---
print("=" * 70)
print("DUPLICATE INVESTIGATION")
print("=" * 70)

# 1. Exact row duplicates in raw df (all columns, incl. PassengerId)
print(f"1. Exact duplicates in raw df (all columns): {df.duplicated().sum()}")

# 2. Duplicate PassengerIds
print(f"2. Duplicate PassengerIds: {df['PassengerId'].duplicated().sum()}")

# 3. Duplicate names (same passenger entered twice)
print(f"3. Duplicate passenger names: {df['Name'].duplicated().sum()}")

# 4. Duplicate rows in cleaned view (same demographics, could be diff. people)
filter_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
dup_in_clean = titanic_clean.duplicated(subset=filter_cols).sum()
print(f"4. Rows with identical demographics in cleaned set: {dup_in_clean}")

# Show sample of demographic duplicate groups
dup_mask = titanic_clean.duplicated(subset=filter_cols, keep=False)
dup_groups = titanic_clean[dup_mask].sort_values(by=filter_cols)
n_unique_groups = dup_groups.drop_duplicates(subset=filter_cols).shape[0]
print(f"   ({dup_mask.sum()} rows across {n_unique_groups} distinct demographic profiles)")
print()
print("Sample duplicate groups (different passengers, same profile):")
print(dup_groups.head(15))
print()

# Conclusion: all rows are distinct passengers, no true duplicates to drop
print("=" * 70)
print("CONCLUSION: No true duplicate records found.")
print("All 111 'duplicate' rows are different passengers with identical")
print("demographics - they represent valid distinct observations. Keep all.")
print("=" * 70)

DUPLICATE INVESTIGATION
1. Exact duplicates in raw df (all columns): 0
2. Duplicate PassengerIds: 0
3. Duplicate passenger names: 0
4. Rows with identical demographics in cleaned set: 118
   (175 rows across 57 distinct demographic profiles)

Sample duplicate groups (different passengers, same profile):
     Survived  Pclass   Sex   Age  SibSp  Parch     Fare Embarked  \
263         0       1  male  40.0      0      0   0.0000        S   
633         0       1  male  40.0      0      0   0.0000        S   
815         0       1  male  40.0      0      0   0.0000        S   
30          0       1  male  40.0      0      0  27.7208        C   
64          0       1  male  40.0      0      0  27.7208        C   
295         0       1  male  40.0      0      0  27.7208        C   
252         0       1  male  62.0      0      0  26.5500        S   
555         0       1  male  62.0      0      0  26.5500        S   
144         0       2  male  18.0      0      0  11.5000        S   
757  

3.3 Correcting Datatypes

In [21]:
# --- Correct datatypes ---
titanic_clean = titanic_clean.copy()

titanic_clean["Survived"] = titanic_clean["Survived"].astype(bool)
titanic_clean["Pclass"] = titanic_clean["Pclass"].astype("category")
titanic_clean["Sex"] = titanic_clean["Sex"].astype("category")
titanic_clean["Embarked"] = titanic_clean["Embarked"].astype("category")
titanic_clean["SibSp"] = titanic_clean["SibSp"].astype(int)
titanic_clean["Parch"] = titanic_clean["Parch"].astype(int)

print("Updated dtypes:")
print(titanic_clean.dtypes)
print()
print(titanic_clean.head(10))

Updated dtypes:
Survived               bool
Pclass             category
Sex                category
Age                 float64
SibSp                 int64
Parch                 int64
Fare                float64
Embarked           category
family_size           int64
traveling_alone        bool
dtype: object

   Survived Pclass     Sex   Age  SibSp  Parch     Fare Embarked  family_size  \
0     False      3    male  22.0      1      0   7.2500        S            2   
1      True      1  female  38.0      1      0  71.2833        C            2   
2      True      3  female  26.0      0      0   7.9250        S            1   
3      True      1  female  35.0      1      0  53.1000        S            2   
4     False      3    male  35.0      0      0   8.0500        S            1   
5     False      3    male  25.0      0      0   8.4583        Q            1   
6     False      1    male  54.0      0      0  51.8625        S            1   
7     False      3    male   2.0      3  

3.4 Cleaning Outliers
 * Scan for any extreme values, if any, then decide case-by-case — don't auto-remove.

In [23]:
# --- Detect outliers in numeric columns ---
num_cols = ["Age", "SibSp", "Parch", "Fare", "family_size"]

print("=" * 70)
print("OUTLIER DETECTION (IQR method)")
print("=" * 70)

for c in num_cols:
    q1 = titanic_clean[c].quantile(0.25)
    q3 = titanic_clean[c].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    mask = (titanic_clean[c] < lo) | (titanic_clean[c] > hi)
    vals = titanic_clean.loc[mask, c]
    print()
    print(f"{c}  (IQR bounds: [{lo:.2f}, {hi:.2f}])")
    print(f"  Outliers: {mask.sum()} / {len(titanic_clean)} ({mask.sum()/len(titanic_clean)*100:.1f}%)")
    #print(f"  Values: {sorted(vals.unique())}")

# --- Case-by-case review ---
print()
print("=" * 70)
print("CASE-BY-CASE REVIEW")
print("=" * 70)

print()
print("Age (58-80): Elderly passengers in 1912 were rare but real.")
print("  No evidence of data-entry error. KEEP ALL.")

print("SibSp (3-8): Large traveling families are documented (e.g. Sage family).")
print("  No evidence of error. KEEP ALL.")

print("Parch (1-6): IQR=0 so any >0 flagged. All are real children/families.")
print("  KEEP ALL.")

print("Fare (66.6-512.33): First-class fares; 512.33 is the Cardeza suite")
print("  (known most expensive ticket). Fare=0 includes Thomas Andrews,")
print("  the ship designer traveling complimentary. All real. KEEP ALL.")

print("family_size (4-11): Derived from SibSp+Parch+1, already validated.")
print("  KEEP ALL.")

print()
print("=" * 70)
print("RESULT: No true outliers removed -- all extreme values are")
print("real observations, not data-entry errors.")
print("=" * 70)

OUTLIER DETECTION (IQR method)

Age  (IQR bounds: [-0.25, 57.75])
  Outliers: 32 / 889 (3.6%)

SibSp  (IQR bounds: [-1.50, 2.50])
  Outliers: 46 / 889 (5.2%)

Parch  (IQR bounds: [0.00, 0.00])
  Outliers: 213 / 889 (24.0%)

Fare  (IQR bounds: [-26.76, 65.66])
  Outliers: 114 / 889 (12.8%)

family_size  (IQR bounds: [-0.50, 3.50])
  Outliers: 91 / 889 (10.2%)

CASE-BY-CASE REVIEW

Age (58-80): Elderly passengers in 1912 were rare but real.
  No evidence of data-entry error. KEEP ALL.
SibSp (3-8): Large traveling families are documented (e.g. Sage family).
  No evidence of error. KEEP ALL.
Parch (1-6): IQR=0 so any >0 flagged. All are real children/families.
  KEEP ALL.
Fare (66.6-512.33): First-class fares; 512.33 is the Cardeza suite
  (known most expensive ticket). Fare=0 includes Thomas Andrews,
  the ship designer traveling complimentary. All real. KEEP ALL.
family_size (4-11): Derived from SibSp+Parch+1, already validated.
  KEEP ALL.

RESULT: No true outliers removed -- all extrem